# 数据合并

In [8]:
import pandas as pd

In [86]:
# 基本数据结构
df_price = pd.read_parquet('raw/300_financial.parquet')
df_finance = pd.read_parquet('raw/300_price.parquet')
print(df_price.head(5))
print(df_finance.head(5))
print(len(df_price), len(df_finance))

         date  WEST_NETPROFIT_FY1_CHG_3M  WEST_NETPROFIT_FTM_CHG_3M  \
0  2016-01-04              -1.129281e+08               3.353710e+08   
1  2016-01-05              -1.129281e+08               3.360947e+08   
2  2016-01-06              -1.129281e+08               3.368184e+08   
3  2016-01-07              -1.129281e+08               3.375421e+08   
4  2016-01-08              -1.129281e+08               3.382658e+08   

   WEST_SALES_FY1_3M  WEST_SALES_FTM_CHG_3M  VAL_AVGPE_SW    PE_TTM  \
0           2.738109           7.326655e+09        6.9459  7.420235   
1           2.738109           7.329704e+09        6.9780  7.466080   
2           2.738109           7.332753e+09        7.0471  7.551219   
3           2.738109           7.335802e+09        6.7295  7.164817   
4           2.738109           7.338851e+09        6.8328  7.282702   

   VAL_STDPE_SW  DFA_TLTSHARE  FA_ROE_AVG  FA_DEBTTOASSET  FA_ORGR_TTM  \
0        0.8854  1.430868e+10     12.3158         93.9541      29.3733  

In [88]:
print(df_price.tail(1))

              date  WEST_NETPROFIT_FY1_CHG_3M  WEST_NETPROFIT_FTM_CHG_3M  \
712107  2026-02-27                        NaN                        NaN   

        WEST_SALES_FY1_3M  WEST_SALES_FTM_CHG_3M  VAL_AVGPE_SW  PE_TTM  \
712107                NaN                    NaN           NaN     NaN   

        VAL_STDPE_SW  DFA_TLTSHARE  FA_ROE_AVG  FA_DEBTTOASSET  FA_ORGR_TTM  \
712107           NaN           NaN         NaN             NaN          NaN   

        FA_CTOP code OUTMESSAGE  
712107      NaN  NAN       None  


In [24]:
# 假设 df_price 是价格表，df_finance 是财务表
# 1. 确保日期列都是 datetime 格式
df_price['date'] = pd.to_datetime(df_price['date'])
df_finance['date'] = pd.to_datetime(df_finance['date'])

# 2. 合并 (取交集）
master_df = pd.merge(df_price, df_finance, on=['date', 'code'], how='inner')

# 3. 排序并填充
# 按照代码和日期排序
master_df = master_df.sort_values(['code', 'date'])

print(master_df.head(5))
print(len(master_df))

        date  WEST_NETPROFIT_FY1_CHG_3M  WEST_NETPROFIT_FTM_CHG_3M  \
0 2016-01-04              -1.129281e+08               3.353710e+08   
1 2016-01-05              -1.129281e+08               3.360947e+08   
2 2016-01-06              -1.129281e+08               3.368184e+08   
3 2016-01-07              -1.129281e+08               3.375421e+08   
4 2016-01-08              -1.129281e+08               3.382658e+08   

   WEST_SALES_FY1_3M  WEST_SALES_FTM_CHG_3M  VAL_AVGPE_SW    PE_TTM  \
0           2.738109           7.326655e+09        6.9459  7.420235   
1           2.738109           7.329704e+09        6.9780  7.466080   
2           2.738109           7.332753e+09        7.0471  7.551219   
3           2.738109           7.335802e+09        6.7295  7.164817   
4           2.738109           7.338851e+09        6.8328  7.282702   

   VAL_STDPE_SW  DFA_TLTSHARE  FA_ROE_AVG  ...  FA_ORGR_TTM  FA_CTOP  \
0        0.8854  1.430868e+10     12.3158  ...      29.3733      NaN   
1       

In [27]:
master_df.to_parquet('raw/300_merged_data.parquet')

# 空值处理
注意这里merged_data_300.parquet是已经合并了价格和财务数据的文件，raw里面是生数据，尽量别动

In [ ]:
# 需要的列
col_hf = ['date', 'code', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME']
col_lf = ['date', 'code', 'FA_ROE_AVG', 'FA_ORGR_TTM', 'FA_DEBTTOASSET', "PE_TTM","WEST_SALES_FTM_CHG_3M", "WEST_NETPROFIT_FTM_CHG_3M"]

In [79]:
df = pd.read_parquet('merged_data_300.parquet')

# 2. 确保按代码和日期排序（非常重要，否则ffill会乱）
df = df.sort_values(['code', 'date'])

# 3. 剔除未上市期的空行
df_cleaned = df.dropna(subset=['OPEN']).copy()

# 4. 对于缺失的数据，使用最近的财务数据进行填充
df_cleaned[col_lf] = df_cleaned.groupby('code')[col_lf].ffill()

# 5. 对于仍然缺失的数据，直接填0
df_cleaned[col_lf] = df_cleaned[col_lf].fillna(0)

# 6. 计算收益率
df_cleaned['target_return'] = df_cleaned.groupby('code')['CLOSE'].shift(-1) / df_cleaned['CLOSE'] - 1

print(df_cleaned.head(5))
print(len(df_cleaned))

        date  WEST_NETPROFIT_FY1_CHG_3M  WEST_NETPROFIT_FTM_CHG_3M  \
0 2016-01-04              -1.129281e+08               3.353710e+08   
1 2016-01-05              -1.129281e+08               3.360947e+08   
2 2016-01-06              -1.129281e+08               3.368184e+08   
3 2016-01-07              -1.129281e+08               3.375421e+08   
4 2016-01-08              -1.129281e+08               3.382658e+08   

   WEST_SALES_FY1_3M  WEST_SALES_FTM_CHG_3M  VAL_AVGPE_SW    PE_TTM  \
0           2.738109           7.326655e+09        6.9459  7.420235   
1           2.738109           7.329704e+09        6.9780  7.466080   
2           2.738109           7.332753e+09        7.0471  7.551219   
3           2.738109           7.335802e+09        6.7295  7.164817   
4           2.738109           7.338851e+09        6.8328  7.282702   

   VAL_STDPE_SW  DFA_TLTSHARE  FA_ROE_AVG  ...  FA_CTOP       code  \
0        0.8854  1.430868e+10     12.3158  ...      NaN  000001.SZ   
1        0.8

In [76]:
# 缺失分析
def analyze_factor_quality(df):
    # 1. 基础缺失率计算
    missing_ratio = df.isnull().mean() * 100
    
    # 2. 统计每只股票的平均有效天数（衡量更新频率）
    update_counts = df.groupby('code').count()
    avg_valid_days = update_counts.mean()
    
    # 3. 统计完全缺失该因子的股票占比（衡量覆盖度）
    zero_coverage_stocks = (update_counts == 0).mean() * 100
    
    # 汇总报告
    report = pd.DataFrame({
        '缺失率 (%)': missing_ratio,
        '覆盖缺失股票占比 (%)': zero_coverage_stocks,
        '平均每只票有效行数': avg_valid_days
    })
    
    return report.sort_values('缺失率 (%)', ascending=False)

# 执行分析
quality_report = analyze_factor_quality(df_cleaned)
print(quality_report)

                              缺失率 (%)  覆盖缺失股票占比 (%)    平均每只票有效行数
OUTMESSAGE                 100.000000    100.000000     0.000000
FA_CTOP                     56.894835      1.388889   947.170139
WEST_SALES_FY1_3M            4.638168      0.347222  2095.430556
WEST_NETPROFIT_FY1_CHG_3M    3.041704      0.000000  2130.510417
VAL_AVGPE_SW                 1.541474      4.513889  2163.475694
VAL_STDPE_SW                 1.541474      4.513889  2163.475694
target_return                0.045509      0.000000  2196.347222
date                         0.000000      0.000000  2197.347222
code                         0.000000           NaN          NaN
WEST_SALES_FTM_CHG_3M        0.000000      0.000000  2197.347222
WEST_NETPROFIT_FTM_CHG_3M    0.000000      0.000000  2197.347222
VOLUME                       0.000000      0.000000  2197.347222
AMT                          0.000000      0.000000  2197.347222
CLOSE                        0.000000      0.000000  2197.347222
OPEN                     

# 数据归一化

In [58]:
# 需要的列
col_hf = ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME']
col_lf = ['FA_ROE_AVG', 'FA_ORGR_TTM', 'FA_DEBTTOASSET', "PE_TTM","WEST_SALES_FTM_CHG_3M", "WEST_NETPROFIT_FTM_CHG_3M"]
target = 'target_return'

In [63]:
from sklearn.preprocessing import StandardScaler

In [ ]:
def winsorize(series):
    # 计算上下分位数
    lower = series.quantile(0.01)
    upper = series.quantile(0.99)
    # 将超出范围的值替换为边界值
    return series.clip(lower, upper)

# 2. 高频量价：执行滚动归一化 (不按行业)
print("正在处理量价指标 (Rolling)...")
rolling_obj = df_cleaned.groupby('code')[col_hf].rolling(window=20)
df_cleaned[col_hf] = (df_cleaned[col_hf] - rolling_obj.mean().reset_index(0, drop=True)) / (rolling_obj.std().reset_index(0, drop=True) + 1e-8)

# 3. 财务指标：执行全局标准化 (保留行业间差异)
print("正在处理财务指标 (Global)...")
df_cleaned[col_lf] = df_cleaned[col_lf].apply(winsorize) # 先去极值
df_cleaned[col_lf] = StandardScaler().fit_transform(df_cleaned[col_lf]) # 后标准化


正在处理量价指标 (Rolling)...
正在处理财务指标 (Global)...


In [85]:
df_cleaned = df_cleaned.dropna(subset=['OPEN']).copy()
df_cleaned = df_cleaned.reset_index(drop=True)
print(df_cleaned.head(5))
print(len(df_cleaned))
df_cleaned.to_parquet('cleaned_300.parquet')

        date  WEST_NETPROFIT_FY1_CHG_3M  WEST_NETPROFIT_FTM_CHG_3M  \
0 2016-01-29              -6.002894e+07                   0.182098   
1 2016-02-01              -6.002894e+07                   0.183956   
2 2016-02-02              -6.002894e+07                   0.184575   
3 2016-02-03              -6.002894e+07                   0.185194   
4 2016-02-04              -6.002894e+07                   0.185813   

   WEST_SALES_FY1_3M  WEST_SALES_FTM_CHG_3M  VAL_AVGPE_SW    PE_TTM  \
0           0.599239               0.260246        6.1239 -0.473854   
1           0.599239               0.260712        6.0169 -0.475900   
2           0.599239               0.260868        6.0808 -0.474365   
3           0.599239               0.261023        6.0249 -0.475389   
4           0.599239               0.261179        6.0988 -0.474365   

   VAL_STDPE_SW  DFA_TLTSHARE  FA_ROE_AVG  ...  FA_CTOP       code  \
0        0.7118  1.430868e+10    0.708562  ...      NaN  000001.SZ   
1        0.7